In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sb
sb.histplot(df["Delivery_Time"])

In [ ]:
# Task 1: Write your code here:
clean_df = df.drop("Order_ID",axis=1)
clean_df.info()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  print()

check_missing_values(clean_df)

#now for the handling
df.head()

clean_df = clean_df.dropna(subset=["Delivery_Time"])

for col in ['Weather','Traffic_Level','Time_of_Day']:
  clean_df[col] = clean_df[col].fillna("unknown")



clean_df["Courier_Experience_yrs"] = clean_df["Courier_Experience_yrs"].fillna(clean_df["Courier_Experience_yrs"].mean())

check_missing_values(clean_df)


In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):

  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(clean_df)
check_duplicates(clean_df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = clean_df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))



for col in categorical_cols:
  le = LabelEncoder()
  clean_df[col] = le.fit_transform(clean_df[col].values)

categorical_cols = clean_df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = clean_df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
clean_df[numerical_cols] = scaler.fit_transform(clean_df[numerical_cols])
clean_df.head()

In [ ]:
# Task 6: Write your code here:
# this is not a classification problem

In [ ]:
# Task 1: Write your code here:
X = clean_df.drop("Delivery_Time", axis=1).astype(float)
y = clean_df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
n_splits = 5 # K=5 Folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
mae_lst = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train
  rfg = RandomForestRegressor()
  print(f"Training...")
  rfg.fit(X_train, y_train)
  y_pred = rfg.predict(X_test)
  mae_lst.append(mean_absolute_error(y_test,y_pred))

print(sum(mae_lst)/len(mae_lst))



In [ ]:
# Task 1: Write your code here:
importance = rfg.feature_importances_
sb.histplot(importance)

In [ ]:
# Task 2: Write your code here:
plt.hist(y_pred)

In [ ]:
# Task Bonus: Write your code here:
# Note: my understanding is that the task wants the average of all MAEs and not the MAEs for each model so that is how i did it

%pip install catboost
from catboost import CatBoostRegressor
models = {"catboost":CatBoostRegressor(),"randomf":RandomForestRegressor()}

n_splits = 5 # K=5 Folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
mae_lst = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train
  for model in models:
    print(f"Training...")
    models[model].fit(X_train, y_train)
    y_pred = models[model].predict(X_test)
    mae_lst.append(mean_absolute_error(y_test,y_pred))

print()
print(sum(mae_lst)/len(mae_lst))